# Toy MCS vs MCS+CSDA-like study

Notebook workflow for running the standalone toy benchmark and plotting inline.

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd

from run_study import run_study
from notebook_helpers import (
    plot_metric_vs_energy_inline,
    plot_metric_vs_fraction_inline,
    plot_reco_scatter_inline,
    plot_rmse_improvement_inline,
)

In [ ]:
event_df, reco_df, metrics_df, like_df, compare_df, out_dir = run_study(
    true_energies=[200.0, 400.0, 800.0, 1200.0, 2000.0],
    contained_fracs=[0.2, 0.4, 0.6, 0.8, 1.0],
    n_events_per_grid=250,
    penalty_mode='soft',
    penalty_sigma_cm=3.0,   # stronger than default for visibility
    penalty_weight=5.0,
)
print('Wrote CSV outputs to:', out_dir)
metrics_df.head()

In [ ]:
# Use f_contained=1.0 first because differences are typically most visible there
plot_metric_vs_energy_inline(metrics_df, metric='bias_mev', f_contained=1.0);
plot_metric_vs_energy_inline(metrics_df, metric='resolution_mev', f_contained=1.0);

# Optional: inspect f_contained=0.6 (often overlaps by construction of lower-bound constraint)
plot_metric_vs_energy_inline(metrics_df, metric='bias_mev', f_contained=0.6);

In [ ]:
plot_metric_vs_fraction_inline(metrics_df, metric='bias_mev', t0_true_mev=800.0);
plot_metric_vs_fraction_inline(metrics_df, metric='resolution_mev', t0_true_mev=800.0);

In [ ]:
plot_reco_scatter_inline(reco_df, method='mcs_only');
plot_reco_scatter_inline(reco_df, method='mcs_csda_like_soft');

In [ ]:
# Quick comparison table
compare = metrics_df[metrics_df['method'].isin(['mcs_only', 'mcs_csda_like_soft'])]\
    .pivot_table(index=['t0_true_mev', 'f_contained'], columns='method', values='rmse_mev')
compare['rmse_improvement_frac'] = (compare['mcs_only'] - compare['mcs_csda_like_soft']) / compare['mcs_only']
compare.head(12)

In [ ]:
# Direct visibility of mcs_csda_like_soft vs mcs_only
compare_df[['t0_true_mev','f_contained','rmse_mev_mcs_only','rmse_mev_mcs_csda_like_soft','frac_rmse_improvement']]\
    .sort_values(['f_contained','t0_true_mev'])
